# Portfolio Dashboard — Notebook Walkthrough

This notebook builds, step by step, the same analytics that power the Streamlit app in `app.py`:

1. Fetch and cache price history with yfinance
2. Candlestick chart with moving averages and volume
3. Portfolio value, returns and risk metrics
4. Allocation donut chart
5. Key fundamentals table

Outside Streamlit there is no `st.cache_data`, so a small TTL cache plays the same role and keeps requests to Yahoo Finance to a minimum.

In [12]:
import time
from functools import wraps

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from plotly.subplots import make_subplots

PERIODS_PER_YEAR = {"1m": 252 * 390, "5m": 252 * 78, "15m": 252 * 26, "1h": 252 * 7, "1d": 252, "1wk": 52, "1mo": 12}

## 1. Configuration and caching

In [13]:
HOLDINGS = {"AAPL": 10, "MSFT": 8, "NVDA": 5, "JPM": 12}
PERIOD = "1y"
INTERVAL = "1d"
MA_WINDOWS = [20, 50]

HISTORY_TTL = 15 * 60
FUNDAMENTALS_TTL = 6 * 60 * 60


def ttl_cache(ttl):
    def decorator(fn):
        store = {}

        @wraps(fn)
        def wrapper(*args):
            hit = store.get(args)
            if hit and time.time() - hit[0] < ttl:
                return hit[1]
            value = fn(*args)
            store[args] = (time.time(), value)
            return value

        wrapper.clear = store.clear
        return wrapper

    return decorator

## 2. Data loaders

In [14]:
@ttl_cache(HISTORY_TTL)
def load_history(ticker, period, interval):
    df = yf.Ticker(ticker).history(period=period, interval=interval, auto_adjust=True)
    if df.empty:
        return df
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df[["Open", "High", "Low", "Close", "Volume"]]


FUNDAMENTAL_FIELDS = {
    "longName": "Name",
    "sector": "Sector",
    "marketCap": "Market Cap",
    "trailingPE": "P/E (TTM)",
    "forwardPE": "Forward P/E",
    "priceToBook": "P/B",
    "trailingEps": "EPS (TTM)",
    "beta": "Beta",
    "profitMargins": "Profit Margin",
    "returnOnEquity": "ROE",
    "debtToEquity": "Debt/Equity",
    "fiftyTwoWeekHigh": "52W High",
    "fiftyTwoWeekLow": "52W Low",
}


@ttl_cache(FUNDAMENTALS_TTL)
def load_fundamentals(ticker):
    try:
        info = yf.Ticker(ticker).info
    except Exception:
        return {}
    row = {label: info.get(key) for key, label in FUNDAMENTAL_FIELDS.items()}
    rate, price = info.get("dividendRate"), info.get("currentPrice") or info.get("regularMarketPrice")
    row["Dividend Yield"] = rate / price if rate and price else None
    return row

In [15]:
histories = {t: load_history(t, PERIOD, INTERVAL) for t in HOLDINGS}
failed = [t for t, h in histories.items() if h.empty]
histories = {t: h for t, h in histories.items() if not h.empty}
if failed:
    print("No data for:", failed)
histories["AAPL"].tail()

,Open,High,Low,Close,Volume
Date,,,,,
2026-10-30,346.720001,349.190002,334.350006,344.630005,67841600
2026-11-02,342.670013,343.799988,335.820007,338.320007,64757700
2026-11-03,325.579987,334.500000,325.329987,331.899994,63160000
2026-11-04,341.570007,354.500000,340.989990,349.510010,89637700
2026-11-05,353.890015,355.380005,342.059998,344.170013,75457100


### Demonstrating the cache

The second call is served from memory, so it returns almost instantly and makes no network request.

In [16]:
start = time.perf_counter()
load_history("AAPL", PERIOD, INTERVAL)
print(f"cached call: {time.perf_counter() - start:.5f}s")

cached call: 0.00047s


## 3. Candlestick chart

In [17]:
def candlestick_figure(df, ticker, ma_windows, hide_weekends=False):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75, 0.25], vertical_spacing=0.03)
    fig.add_trace(
        go.Candlestick(x=df.index, open=df["Open"], high=df["High"], low=df["Low"], close=df["Close"], name=ticker),
        row=1, col=1,
    )
    for w in ma_windows:
        fig.add_trace(
            go.Scatter(x=df.index, y=df["Close"].rolling(w).mean(), name=f"MA{w}", line=dict(width=1.3)),
            row=1, col=1,
        )
    colors = np.where(df["Close"] >= df["Open"], "#26a69a", "#ef5350")
    fig.add_trace(go.Bar(x=df.index, y=df["Volume"], marker_color=colors, name="Volume", showlegend=False), row=2, col=1)
    if hide_weekends:
        fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])
    fig.update_layout(height=640, xaxis_rangeslider_visible=False, margin=dict(l=10, r=10, t=30, b=10),
                      legend=dict(orientation="h", y=1.02))
    return fig


ticker = next(iter(histories))
candlestick_figure(histories[ticker], ticker, MA_WINDOWS).show()

## 4. Portfolio value and performance

The portfolio is buy-and-hold: share counts stay constant over the period.

In [18]:
def build_close_frame(histories):
    closes = pd.DataFrame({t: h["Close"] for t, h in histories.items()})
    return closes.sort_index().ffill().dropna()


closes = build_close_frame(histories)
shares = pd.Series({t: HOLDINGS[t] for t in closes.columns})
position_values = closes * shares
portfolio_values = position_values.sum(axis=1)
portfolio_values.tail()

Date
2026-10-30    12689.259933
2026-11-02    12537.419861
2026-11-03    12576.130005
2026-11-04    13167.469849
2026-11-05    12685.220062
dtype: float64

In [19]:
def performance_metrics(values, periods_per_year):
    returns = values.pct_change().dropna()
    drawdown = values / values.cummax() - 1
    years = max((values.index[-1] - values.index[0]).days / 365.25, 1e-9)
    total = values.iloc[-1] / values.iloc[0] - 1
    vol = returns.std() * np.sqrt(periods_per_year) if len(returns) > 1 else np.nan
    cagr = (1 + total) ** (1 / years) - 1 if years >= 1 else np.nan
    sharpe = returns.mean() * periods_per_year / vol if len(returns) > 1 and vol else np.nan
    return {
        "Total return": total,
        "CAGR": cagr,
        "Volatility": vol,
        "Sharpe (rf=0)": sharpe,
        "Max drawdown": drawdown.min(),
    }


metrics = performance_metrics(portfolio_values, PERIODS_PER_YEAR[INTERVAL])
pd.Series(metrics, name="Value").to_frame().style.format("{:.2%}", subset=pd.IndexSlice[["Total return", "CAGR", "Volatility", "Max drawdown"], :], na_rep="—").format("{:.2f}", subset=pd.IndexSlice[["Sharpe (rf=0)"], :], na_rep="—")

,Value
Total return,12.65%
CAGR,—
Volatility,17.55%
Sharpe (rf=0),0.77
Max drawdown,-17.53%


In [20]:
def performance_figure(values, closes):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.06,
                        subplot_titles=("Growth of 100 (portfolio vs holdings)", "Drawdown"))
    fig.add_trace(go.Scatter(x=values.index, y=values / values.iloc[0] * 100, name="Portfolio",
                             line=dict(width=3, color="#1f77b4")), row=1, col=1)
    for t in closes.columns:
        fig.add_trace(go.Scatter(x=closes.index, y=closes[t] / closes[t].iloc[0] * 100, name=t,
                                 line=dict(width=1), opacity=0.6), row=1, col=1)
    drawdown = values / values.cummax() - 1
    fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown, fill="tozeroy", name="Drawdown",
                             line=dict(color="#ef5350"), showlegend=False), row=2, col=1)
    fig.update_yaxes(tickformat=".0%", row=2, col=1)
    fig.update_layout(height=620, margin=dict(l=10, r=10, t=40, b=10), hovermode="x unified")
    return fig


performance_figure(portfolio_values, closes).show()

## 5. Allocation

In [21]:
latest_values = position_values.iloc[-1]
weights = latest_values / latest_values.sum()

fig = go.Figure(go.Pie(labels=weights.index, values=weights.values, hole=0.45, textinfo="label+percent"))
fig.update_layout(height=480, margin=dict(l=10, r=10, t=10, b=10))
fig.show()

pd.DataFrame({"Shares": shares, "Price": closes.iloc[-1], "Value": latest_values, "Weight": weights}).style.format(
    {"Shares": "{:,.2f}", "Price": "${:,.2f}", "Value": "${:,.0f}", "Weight": "{:.1%}"}
)

,Shares,Price,Value,Weight
AAPL,10.00,$344.17,"$3,442",27.1%
MSFT,8.00,$535.93,"$4,287",33.8%
NVDA,5.00,$181.60,$908,7.2%
JPM,12.00,$337.34,"$4,048",31.9%


## 6. Fundamentals

In [22]:
rows = {t: load_fundamentals(t) for t in histories}
fundamentals = pd.DataFrame.from_dict(rows, orient="index").reindex(
    columns=[*FUNDAMENTAL_FIELDS.values(), "Dividend Yield"]
)

pct = ["Profit Margin", "ROE", "Dividend Yield"]
ratio = ["P/E (TTM)", "Forward P/E", "P/B", "EPS (TTM)", "Beta", "Debt/Equity", "52W High", "52W Low"]
display = fundamentals.copy()
for c in pct:
    display[c] = display[c].map(lambda v: f"{v:.2%}" if pd.notna(v) else "—")
for c in ratio:
    display[c] = display[c].map(lambda v: f"{v:,.2f}" if pd.notna(v) else "—")
display["Market Cap"] = display["Market Cap"].map(lambda v: f"{v / 1e9:,.1f}B" if pd.notna(v) else "—")
display.fillna("—")

,Name,Sector,Market Cap,P/E (TTM),Forward P/E,P/B,EPS (TTM),Beta,Profit Margin,ROE,Debt/Equity,52W High,52W Low,Dividend Yield
AAPL,Apple Inc.,Technology,"5,001.5B",38.33,32.32,44.62,8.98,1.09,27.60%,142.26%,84.29,355.38,243.42,0.31%
MSFT,Microsoft Corporation,Technology,"3,979.9B",27.97,22.16,8.46,19.16,1.04,41.17%,34.22%,28.46,559.15,349.20,0.73%
NVDA,NVIDIA Corporation,Technology,"4,385.1B",22.96,11.41,19.15,7.91,2.25,63.66%,117.21%,16.97,243.37,164.27,0.55%
JPM,JPMorgan Chase & Co.,Financial Services,895.8B,14.15,13.19,2.55,23.84,1.02,34.59%,18.01%,—,366.50,279.10,1.96%


## 7. Intraday candles

Yahoo limits intraday history: 1m about 7 days, 5m and 15m about 60 days, 1h about 730 days.

In [23]:
intraday = load_history(ticker, "5d", "5m")
candlestick_figure(intraday, ticker, [20], hide_weekends=True).show()

## 8. Live polling and recording

`recorder.py` (in the same folder) fetches the latest 1-minute bar for every ticker in one batched request and upserts it into SQLite, keyed by ticker and bar timestamp. Re-polling the same bar updates its row instead of adding a duplicate.

When the market is closed the latest bar does not change, so repeated polls update the same rows.

In [24]:
from recorder import DB_PATH, fetch_latest, read_ticks, save_ticks

POLLS = 3
POLL_SECONDS = 30

for i in range(POLLS):
    quotes = fetch_latest(list(HOLDINGS))
    saved = save_ticks(quotes)
    print(f"poll {i + 1}/{POLLS}: saved {saved} quotes")
    if i < POLLS - 1:
        time.sleep(POLL_SECONDS)

quotes

poll 1/3: saved 4 quotes
poll 2/3: saved 4 quotes
poll 3/3: saved 4 quotes


,ticker,bar_ts,price,volume,session_open
0,AAPL,2026-11-05T20:59:00+00:00,344.309998,1569783.0,354.250000
1,MSFT,2026-11-05T20:59:00+00:00,535.659973,663168.0,555.754028
2,NVDA,2026-11-05T20:59:00+00:00,181.570007,4082940.0,195.179993
3,JPM,2026-11-05T20:59:00+00:00,337.410004,247810.0,358.635010


## 9. Review recorded ticks

Ticks recorded by the app or by `python recorder.py` land in the same database.

In [25]:
ticks = read_ticks(list(HOLDINGS))
print(f"{len(ticks):,} rows in {DB_PATH}")

fig = go.Figure()
for t, g in ticks.groupby("ticker"):
    fig.add_trace(go.Scatter(x=g["bar_ts"], y=g["price"] / g["price"].iloc[0] * 100, name=t,
                             mode="lines+markers", marker=dict(size=4)))
fig.update_layout(height=420, hovermode="x unified", yaxis_title="Recorded price (first tick = 100)",
                  margin=dict(l=10, r=10, t=10, b=10))
fig.show()

ticks.tail()

4 rows in C:\Users\Lenovo\Documents\School Staff\personal projects\Yfinance\data\ticks.db


,ticker,bar_ts,price,volume,recorded_at
0,AAPL,2026-11-05 20:59:00+00:00,344.309998,1569783.0,2026-11-06T09:34:18.140832+00:00
1,JPM,2026-11-05 20:59:00+00:00,337.410004,247810.0,2026-11-06T09:34:18.140832+00:00
2,MSFT,2026-11-05 20:59:00+00:00,535.659973,663168.0,2026-11-06T09:34:18.140832+00:00
3,NVDA,2026-11-05 20:59:00+00:00,181.570007,4082940.0,2026-11-06T09:34:18.140832+00:00


## Notes

- yfinance is an unofficial wrapper around Yahoo Finance; fields can be missing and data is not guaranteed to be accurate. Data can be delayed by about 15 minutes, so the live feed is near-live, not streaming.
- To force a refresh in this notebook, call `load_history.clear()` and `load_fundamentals.clear()`.
- For continuous recording without a browser or notebook, run `python recorder.py --tickers AAPL,MSFT --interval 60`.
- Run the full interactive app with `streamlit run app.py`.